[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/divyamohan1993/dip-practical/blob/main/Practical_7.ipynb)

> **Run in Google Colab:** Click the badge above to open this notebook in Google Colab. The dataset downloads automatically — no setup needed.

<style>
/* =====================================================================
   Practical Handbook — Uniform Print Formatting (CSU2543 Digital Image Processing)
   Sections are color-coded: Aim (blue), Theory (purple), Code (green),
   Output (amber), Analysis (maroon). Code blocks are uniform across all
   practicals. Page layout: 1-inch margins, justified text, Times New Roman.
   ===================================================================== */
@page { margin: 1in; }
@media print {
  body, .jp-Notebook, .jupyter-renderer { background: white !important; }
  .jp-CodeMirrorEditor, .CodeMirror { font-size: 10pt !important; }
  .jp-Cell-inputCollapser, .jp-Cell-outputCollapser, .jp-Toolbar { display: none !important; }
  .dip-section-marker { page-break-inside: avoid; }
}

.jp-RenderedHTMLCommon, .jp-RenderedMarkdown {
  font-family: 'Times New Roman', Georgia, serif;
  font-size: 12pt;
  line-height: 1.55;
  text-align: justify;
}

/* Section markers — colored heading bars */
.dip-section-marker {
  display: block;
  font-weight: bold;
  font-size: 1.35em;
  padding: 0.45em 0.7em;
  margin: 1.1em 0 0.6em 0;
  border-left: 6px solid;
  border-radius: 3px;
  letter-spacing: 0.02em;
  page-break-after: avoid;
}
.dip-section-aim      { color: #003c8f; background: #e3f2fd; border-color: #1565c0; }
.dip-section-theory   { color: #4a148c; background: #f3e5f5; border-color: #6a1b9a; }
.dip-section-code     { color: #1b5e20; background: #e8f5e9; border-color: #2e7d32; }
.dip-section-output   { color: #e65100; background: #fff3e0; border-color: #ef6c00; }
.dip-section-analysis { color: #b71c1c; background: #ffebee; border-color: #c62828; }

/* Sub-section headings within a section ("Part 1", "Part 2", ...) */
.dip-subsection {
  font-weight: 600;
  font-size: 1.1em;
  margin: 0.9em 0 0.4em 0;
  color: #2e7d32;
  border-bottom: 1px solid #c8e6c9;
  padding-bottom: 0.15em;
  page-break-after: avoid;
}

/* Code cells — uniform JetBrains Mono / Source Code Pro across all practicals */
.jp-CodeMirrorEditor, .jp-CodeCell .jp-InputArea-editor,
.CodeMirror, pre, .highlight, code {
  font-family: 'JetBrains Mono', 'Source Code Pro', Consolas, 'Courier New', monospace !important;
}
.jp-CodeCell .jp-InputArea-editor, .CodeMirror, pre {
  font-size: 10pt !important;
  background: #f6f8fa !important;
  border-left: 3px solid #2e7d32 !important;
  border-radius: 0 !important;
  padding: 8px 12px !important;
}
code { background: #f6f8fa; padding: 1px 5px; border-radius: 2px; font-size: 0.95em; }

/* Output cells — amber tint, matching the Output section colour */
.jp-OutputArea-output {
  border-left: 3px solid #ef6c00 !important;
  background: #fffaf3 !important;
  padding: 6px 10px !important;
}

/* Analysis question lists */
.dip-analysis-list { padding-left: 1.4em; }
.dip-analysis-list li { margin-bottom: 0.5em; text-align: justify; }
</style>

# Practical 7: 2D Correlation and Convolution

<span class="dip-section-marker dip-section-aim">1. Aim</span>

To implement and analyze the two fundamental linear spatial operations of digital image processing — correlation and convolution — from first principles, demonstrate the impulse-response interpretation that distinguishes them, and apply standard kernels (averaging, Laplacian, Sobel) to a real image.

<span class="dip-section-marker dip-section-theory">2. Description / Theory</span>

Let $f$ be an $M\times N$ image and $w$ an $m\times n$ kernel with $m=2a+1, n=2b+1$.

$$ (w \star f)(x,y) = \sum_{s=-a}^{a}\sum_{t=-b}^{b} w(s,t)\,f(x+s,y+t) \qquad\text{correlation} $$

$$ (w * f)(x,y) = \sum_{s=-a}^{a}\sum_{t=-b}^{b} w(s,t)\,f(x-s,y-t) \qquad\text{convolution} $$

**Identity:** $w * f \;=\; \mathrm{rot}_{180}(w) \star f$. Consequently a unit-impulse input writes $\mathrm{rot}_{180}(w)$ into the correlation output and $w$ itself into the convolution output—this is the operational definition. The two operations coincide whenever $w$ is symmetric under 180° rotation (box, Gaussian, standard Laplacian); they differ for asymmetric kernels (Sobel, Prewitt, motion blur).

<span class="dip-section-marker dip-section-code">3. Code</span>

## Code

### Setup
Import dependencies and (for the image-based parts) auto-download the Gonzalez & Woods Chapter 2 dataset.

In [ ]:
# Install dependencies (uncomment for Google Colab)
# !pip install opencv-python-headless matplotlib numpy

import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

In [ ]:
# === AUTO-DOWNLOAD DATASET (works in Google Colab and locally) ===
import os, urllib.request, zipfile

CHAPTER = "CH02"  # Chapter 2: Digital Image Fundamentals
DATASET_PATH = f"datasets/{CHAPTER}/"
DOWNLOAD_BASE = "https://www.imageprocessingplace.com/downloads_V3/dip3e_downloads/dip3e_book_images"

if not os.path.exists(DATASET_PATH) or not any(f.endswith('.tif') for f in os.listdir(DATASET_PATH)):
    zip_name = f"DIP3E_{CHAPTER}_Original_Images.zip"
    url = f"{DOWNLOAD_BASE}/{zip_name}"
    print(f"Downloading {CHAPTER} dataset from imageprocessingplace.com...")
    urllib.request.urlretrieve(url, "chapter.zip")
    os.makedirs(DATASET_PATH, exist_ok=True)
    with zipfile.ZipFile("chapter.zip", "r") as z:
        for f in z.namelist():
            if f.lower().endswith(".tif"):
                fname = os.path.basename(f)
                if fname:
                    with z.open(f) as src, open(os.path.join(DATASET_PATH, fname), "wb") as dst:
                        dst.write(src.read())
    os.remove("chapter.zip")
    print(f"Downloaded {len([f for f in os.listdir(DATASET_PATH) if f.endswith('.tif')])} images")
else:
    print(f"Dataset ready: {len([f for f in os.listdir(DATASET_PATH) if f.endswith('.tif')])} images")

### Part 1: Manual Correlation and Convolution
Define `correlate2d` and `convolve2d` from the summation definitions, with explicit zero padding and a `mode` argument to select `same` or `full` output extent.

In [ ]:
def correlate2d(f, w, mode='same'):
    """Direct 2D correlation: (w * f)(x,y) = sum_{s,t} w(s,t) f(x+s, y+t).

    mode = 'same' returns an array of the same size as f.
    mode = 'full' returns the full sliding extent (M+m-1, N+n-1).
    """
    f = np.asarray(f, dtype=np.float64)
    w = np.asarray(w, dtype=np.float64)
    fh, fw = f.shape
    wh, ww = w.shape
    a, b = wh // 2, ww // 2
    padded = np.pad(f, ((a, a), (b, b)), mode='constant', constant_values=0)
    if mode == 'same':
        out = np.zeros((fh, fw), dtype=np.float64)
        for x in range(fh):
            for y in range(fw):
                out[x, y] = np.sum(w * padded[x:x+wh, y:y+ww])
        return out, padded
    elif mode == 'full':
        big = np.pad(f, ((wh-1, wh-1), (ww-1, ww-1)), mode='constant', constant_values=0)
        H, W = fh + wh - 1, fw + ww - 1
        out = np.zeros((H, W), dtype=np.float64)
        for x in range(H):
            for y in range(W):
                out[x, y] = np.sum(w * big[x:x+wh, y:y+ww])
        return out, padded
    else:
        raise ValueError("mode must be 'same' or 'full'")

def convolve2d(f, w, mode='same'):
    """Direct 2D convolution = correlation with w rotated 180 degrees."""
    w_rot = np.rot90(w, 2)
    return correlate2d(f, w_rot, mode=mode)

print('correlate2d and convolve2d defined.')

### Part 2: Impulse-Response Demonstration
Place a unit impulse at the centre of a $3\times3$ image and operate with an asymmetric $3\times3$ kernel. The two operations must give visibly different outputs: correlation places a flipped copy of $w$ at the impulse, while convolution places $w$ itself.

In [ ]:
f = np.array([
    [0, 0, 0],
    [0, 1, 0],
    [0, 0, 0],
])

w = np.array([
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9],
])

corr_same, padded_f = correlate2d(f, w, mode='same')
conv_same, _        = convolve2d(f, w, mode='same')
corr_full, _        = correlate2d(f, w, mode='full')
conv_full, _        = convolve2d(f, w, mode='full')

print('--- Inputs ---')
print('Image f =\n', f)
print('Kernel w =\n', w)
print('Rotated kernel rot180(w) =\n', np.rot90(w, 2))
print('Zero-padded f =\n', padded_f.astype(int))

print('\n--- Correlation (w * f) ---')
print('same :\n', corr_same.astype(int))
print('full :\n', corr_full.astype(int))

print('\n--- Convolution (w (*) f) ---')
print('same :\n', conv_same.astype(int))
print('full :\n', conv_full.astype(int))

print('\nObservation: at the impulse, correlation reproduces rot180(w);')
print('             convolution reproduces w itself.')

### Part 3: Side-by-Side Visualisation of the Impulse Response
Render the inputs and the two output matrices as heatmaps so the difference between correlation and convolution is unmistakable to the eye.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 8))

panels_top = [
    ('Image f (impulse)', f),
    ('Kernel w', w),
    ('rot180(w)', np.rot90(w, 2)),
]
panels_bot = [
    ('Padded f', padded_f.astype(int)),
    ('Correlation (same)', corr_same.astype(int)),
    ('Convolution (same)', conv_same.astype(int)),
]

for ax, (title, mat) in zip(axes[0], panels_top):
    im = ax.imshow(mat, cmap='viridis')
    ax.set_title(title)
    for (i, j), v in np.ndenumerate(mat):
        ax.text(j, i, str(int(v)), ha='center', va='center', color='white', fontsize=11, fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])

for ax, (title, mat) in zip(axes[1], panels_bot):
    im = ax.imshow(mat, cmap='viridis')
    ax.set_title(title)
    for (i, j), v in np.ndenumerate(mat):
        ax.text(j, i, str(int(v)), ha='center', va='center', color='white', fontsize=10, fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Impulse Response: Correlation vs Convolution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Part 4: Application to a Real Image
Apply three standard kernels — a $3\times3$ box (averaging), a Laplacian (sharpening), and a horizontal Sobel (edge detection) — to a grayscale image using the manual `convolve2d` function, and display the original alongside each filtered output.

In [ ]:
selected_image = "Fig0222(b)(cameraman).tif"
img = cv2.imread(os.path.join(DATASET_PATH, selected_image), cv2.IMREAD_GRAYSCALE)

print(f"Image:  {selected_image}")
print(f"Shape:  {img.shape}")
print(f"Range:  [{img.min()}, {img.max()}]")

# Standard kernels
box = np.ones((3, 3), dtype=np.float64) / 9.0
lap = np.array([[ 0, -1,  0],
                [-1,  4, -1],
                [ 0, -1,  0]], dtype=np.float64)
sobel_x = np.array([[-1, 0, 1],
                    [-2, 0, 2],
                    [-1, 0, 1]], dtype=np.float64)

out_box, _ = convolve2d(img.astype(np.float64), box,     mode='same')
out_lap, _ = convolve2d(img.astype(np.float64), lap,     mode='same')
out_sob, _ = convolve2d(img.astype(np.float64), sobel_x, mode='same')

def to_uint8(a):
    a = a - a.min()
    a = 255.0 * a / max(a.max(), 1e-9)
    return a.astype(np.uint8)

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
axes[0].imshow(img, cmap='gray', vmin=0, vmax=255); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(np.clip(out_box, 0, 255).astype(np.uint8), cmap='gray', vmin=0, vmax=255)
axes[1].set_title('Box 3x3 (smoothing)'); axes[1].axis('off')
axes[2].imshow(to_uint8(out_lap), cmap='gray'); axes[2].set_title('Laplacian (edges, normalised)'); axes[2].axis('off')
axes[3].imshow(to_uint8(out_sob), cmap='gray'); axes[3].set_title('Sobel-X (vertical edges, normalised)'); axes[3].axis('off')
plt.suptitle('Manual convolve2d: Three Standard Kernels on Cameraman', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Part 5: Verification Against `cv2.filter2D`
OpenCV's `filter2D` performs **correlation**, not convolution. To compare against our manual `convolve2d` we must pre-rotate the kernel before calling it. The maximum absolute difference between the two outputs should be effectively zero (limited only by floating-point rounding).

In [ ]:
kernel = sobel_x  # asymmetric, so corr != conv

manual_conv, _ = convolve2d(img.astype(np.float64), kernel, mode='same')
cv_conv = cv2.filter2D(img.astype(np.float64), ddepth=-1,
                       kernel=np.rot90(kernel, 2),
                       borderType=cv2.BORDER_CONSTANT)
manual_corr, _ = correlate2d(img.astype(np.float64), kernel, mode='same')
cv_corr = cv2.filter2D(img.astype(np.float64), ddepth=-1,
                       kernel=kernel,
                       borderType=cv2.BORDER_CONSTANT)

print(f"max |manual_conv - cv_conv| = {np.max(np.abs(manual_conv - cv_conv)):.6f}")
print(f"max |manual_corr - cv_corr| = {np.max(np.abs(manual_corr - cv_corr)):.6f}")
print(f"max |conv - corr|           = {np.max(np.abs(manual_conv - manual_corr)):.6f}")
print("\nA non-zero conv-vs-corr gap confirms the kernel is asymmetric.")

## Output

All output (printed values, matrix prints, and rendered figures) appears immediately below the corresponding code cells when the notebook is executed top-to-bottom.

<span class="dip-section-marker dip-section-output">4. Output</span>

*All output (printed values, computed statistics, and rendered figures) appears immediately below the corresponding code cells when this notebook is executed top-to-bottom.*

<span class="dip-section-marker dip-section-analysis">5. Analysis / Conclusion</span>

**1. Why does correlation produce a flipped copy of $w$ when $f$ is a unit impulse, while convolution produces $w$ itself?**  
Correlation evaluates $\sum w(s,t)\,f(x+s,y+t)$. With $f$ a unit impulse at the origin, only the term with $f=1$ survives, and the sum picks the kernel value at the position **opposite** to the offset, which leaves $w$ written into the output rotated by 180°. Convolution introduces a sign flip in the indices ($f(x-s,y-t)$), which is exactly equivalent to first rotating $w$ by 180° and then correlating — the two rotations cancel, and $w$ appears in the output untransformed. This is the operational definition of the kernel as the *impulse response* of the filter under convolution.

**2. When are correlation and convolution numerically identical?**  
Whenever the kernel is symmetric under 180° rotation, i.e., $w(s,t)=w(-s,-t)$. Box filters, Gaussians, and the standard Laplacian all satisfy this and may be implemented with either operation interchangeably. Asymmetric kernels — Sobel, Prewitt, motion-blur, derivative-of-Gaussian — give different results, and the choice of operation must be stated explicitly.

**3. What is the role of zero padding, and what alternatives exist?**  
Without padding the kernel cannot be centred on border pixels, so the output is smaller than the input. Zero padding extends $f$ by $a$ rows and $b$ columns of zeros, restoring the original output size. Zeros bias the response near the border toward zero, which can introduce a dark frame; alternatives that avoid this artefact include replicate (`BORDER_REPLICATE`), reflect (`BORDER_REFLECT`), and wrap (`BORDER_WRAP`) — each appropriate for different signal assumptions.

**4. What is the computational cost, and how is it reduced in practice?**  
A direct $m\times n$ kernel applied to an $M\times N$ image costs $O(MNmn)$ multiply-adds. Two reductions are standard: (i) **separability** — if $w = u\,v^{T}$, the 2D operation becomes two 1D passes for $O(MN(m+n))$ total cost; box and Gaussian kernels are separable. (ii) **FFT** — for large kernels, transforming both arrays and multiplying in the frequency domain costs $O(MN\log MN)$, breaking even with the direct method around $m\!\sim\!7$ for typical image sizes.